### Environment Setup

In [1]:
import math
import random
import matplotlib.pyplot as plt
%matplotlib inline

random.seed(1337)

### The Core Value Class (Autograd Engine)

What this cell does: 
Defines the primary building block of Micrograd. The Value class wraps a raw scalar (data), tracks its partial derivative relative to the loss (grad), and builds a Directed Acyclic Graph (DAG) using _prev.Key 

Inner Mechanics to Note:
_backward() Closures: Each mathematical operation (+, *, tanh, **) defines its own local chain-rule derivative using a nested _backward() function. <br>
Gradient Accumulation (+=): Gradients use += instead of = because a variable can be reused in multiple paths through the computational graph (e.g., $f = x + x$). <br>
Topological Sort (backward()): Orders the graph nodes sequentially from output to inputs, ensuring every upstream node passes its complete gradient down before its children evaluate theirs. <br>

In [2]:
class Value:
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data
        self.grad = 0.0                  # Stores dL/d(this_node)
        self._backward = lambda: None   # Holds local chain-rule closure
        self._prev = set(_children)     # Tracks child nodes in computational DAG
        self._op = _op                   # Operational string symbol (+, *, etc.)
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        
        def _backward():
            # Addition passes upstream gradient equally to both inputs
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        
        def _backward():
            # Multiplication swaps operands and scales by upstream gradient
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "Supporting int/float powers only"
        out = Value(self.data**other, (self,), f'**{other}')

        def _backward():
            # Power rule: d/dx(x^n) = n * x^(n-1)
            self.grad += (other * self.data**(other - 1)) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        x = self.data
        t = (math.exp(2*x) - 1)/(math.exp(2*x) + 1)
        out = Value(t, (self,), 'tanh')
        
        def _backward():
            # Local derivative of tanh(x) is (1 - tanh(x)^2)
            self.grad += (1.0 - t**2) * out.grad
        out._backward = _backward
        return out

    def backward(self):
        # Build topological sort to sequence nodes backward
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)

        # Base case: dL/dL = 1.0
        self.grad = 1.0
        
        # Execute backward pass in reverse topological order
        for node in reversed(topo):
            node._backward()

    # Operator overloads to support standard Python syntax (e.g., a - b, 2 * a)
    def __neg__(self): return self * -1
    def __radd__(self, other): return self + other
    def __sub__(self, other): return self + (-other)
    def __rsub__(self, other): return other + (-self)
    def __rmul__(self, other): return self * other
    def __truediv__(self, other): return self * (other**-1)

### Neural Network Modular Abstractions

What this cell does: 
Builds standard neural network components bottom-up using object-oriented abstractions similar to torch.nn:
* Neuron: Represents $y = \tanh(\sum w_i x_i + b)$. Initializes weights randomly between -1 and 1.
* Layer: A collection of Neuron instances running in parallel on identical inputs.
* MLP (Multi-Layer Perceptron): Chains multiple Layer instances sequentially.Crucial Function: The parameters() method recursively collects all Value weights and biases across the network so the optimizer can loop through them.

In [3]:
class Neuron:
    def __init__(self, nin):
        # Initialize weights and bias randomly in [-1.0, 1.0]
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1, 1))

    def __call__(self, x):
        # Compute dot product: w * x + b
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        return act.tanh()

    def parameters(self):
        return self.w + [self.b]


class Layer:
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs

    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()]


class MLP:
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

The 4-Step Optimization Loop
What this cell does: Trains our custom MLP on a small binary dataset. This demonstrates the standard neural network optimization sequence executed on every training step:
* Forward Pass: Compute network predictions and evaluate overall Mean Squared Error (MSE) Loss.
* Zero Gradients (p.grad = 0.0): Reset accumulated gradients from previous steps to prevent infinite gradient growth.
* Backward Pass (loss.backward()): Populate .grad on all parameter nodes via automatic differentiation.
* Gradient Step (SGD): Nudge parameter values in the opposite direction of the gradient ($w = w - \eta \cdot \nabla w$).

In [4]:
# Dataset: 4 samples, 3 input features each
xs = [
  [2.0, 3.0, -1.0],
  [3.0, -1.0, 0.5],
  [0.5, 1.0, 1.0],
  [1.0, 1.0, -1.0],
]
ys = [1.0, -1.0, -1.0, 1.0] # Target outputs

# Initialize MLP (3 inputs -> 4 hidden -> 4 hidden -> 1 output)
model = MLP(3, [4, 4, 1])

learning_rate = 0.05

for step in range(50):
    # Step 1: Forward Pass
    ypred = [model(x) for x in xs]
    loss = sum((yout - ygt)**2 for ygt, yout in zip(ys, ypred))
    
    # Step 2: Zero Gradients (CRITICAL STEP)
    for p in model.parameters():
        p.grad = 0.0
        
    # Step 3: Backward Pass
    loss.backward()
    
    # Step 4: Stochastic Gradient Descent (SGD) Parameter Update
    for p in model.parameters():
        p.data += -learning_rate * p.grad
        
    if step % 10 == 0 or step == 49:
        print(f"Step {step:2d} | Loss: {loss.data:.6f}")

Step  0 | Loss: 6.005403
Step 10 | Loss: 0.505520
Step 20 | Loss: 0.053542
Step 30 | Loss: 0.025741
Step 40 | Loss: 0.016607
Step 49 | Loss: 0.012490


### Verification Against PyTorch (torch.autograd)
What this cell does: Re-implements the exact same computational node values in PyTorch using torch.tensor(..., requires_grad=True). Calling o.backward() in PyTorch performs the exact same mathematical graph evaluation as Value.backward(), verifying that Micrograd produces identical gradients.

In [6]:
import torch

# Initialize PyTorch tensors with float64 precision matching scalar operations
x1 = torch.tensor([2.0], dtype=torch.float64, requires_grad=True)
x2 = torch.tensor([0.0], dtype=torch.float64, requires_grad=True)
w1 = torch.tensor([-3.0], dtype=torch.float64, requires_grad=True)
w2 = torch.tensor([1.0], dtype=torch.float64, requires_grad=True)
b  = torch.tensor([6.8813735870195432], dtype=torch.float64, requires_grad=True)

# Forward pass
n = x1*w1 + x2*w2 + b
o = torch.tanh(n)

# Backward pass
o.backward()

print("--- PyTorch Autograd Verification ---")
print("Output Value :", o.item())
print("x1.grad      :", x1.grad.item())
print("w1.grad      :", w1.grad.item())
print("x2.grad      :", x2.grad.item())
print("w2.grad      :", w2.grad.item())

--- PyTorch Autograd Verification ---
Output Value : 0.7071067811865476
x1.grad      : -1.4999999999999998
w1.grad      : 0.9999999999999999
x2.grad      : 0.49999999999999994
w2.grad      : 0.0
